In [ ]:
# Install required packages
!pip install python-jose cryptography langchain-core langchain-community langchain-openai langchain-text-splitters langchain-experimental faiss-cpu python-dotenv pyyaml numpy pandas pypdf PyMuPDF rank-bm25

# A7 – Advanced RAG (Grounding + Citation + Abstention)

- **Experiment ID:** A7_ADVANCED
- **Adapted from:** reliable_rag.ipynb
- **Purpose:** Full pipeline: hybrid + RRF + rerank + grounded prompt with [S1] citation + abstention.
- **Key comparison:** A0 (Naive) vs A7 (Advanced)

In [ ]:
import sys, json
import numpy as np, pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)
EXPERIMENT_ID = 'A7_ADVANCED'
NOTEBOOK = '08_advanced_rag.ipynb'
SEED = config['seed']
CONFIG_HASH = config['_config_hash']
FINAL_TOP_K = config['retrieval']['final_top_k']
CANDIDATE_TOP_K = config['retrieval']['candidate_top_k']
RRF_K = config['retrieval']['rrf_k']

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f'LLM OK: {llm.invoke("hi").content[:30]}')

## Build Indexes

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

raw_data_path = PROJECT_ROOT / config['paths']['raw_data']
documents = []
for pdf_file in raw_data_path.glob('*.pdf'):
    documents.extend(PyPDFLoader(str(pdf_file)).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
for c in chunks: c.page_content = c.page_content.replace('\t', ' ')

vectorstore = FAISS.from_documents(chunks, embeddings)
chunk_texts = [c.page_content for c in chunks]
bm25 = BM25Okapi([t.lower().split() for t in chunk_texts])

with open(PROJECT_ROOT / config['paths']['questions'], 'r') as f:
    eval_questions = json.load(f)
print(f'Ready: {len(chunks)} chunks, {len(eval_questions)} questions')

## Advanced Retrieve (Hybrid RRF)

In [ ]:
def advanced_retrieve(question):
    dense_docs = vectorstore.similarity_search(question, k=CANDIDATE_TOP_K)
    bm25_scores = bm25.get_scores(question.lower().split())
    bm25_top = np.argsort(bm25_scores)[::-1][:CANDIDATE_TOP_K]
    rrf = {}
    for r, d in enumerate(dense_docs, 1):
        rrf[d.page_content[:80]] = {'s': 1/(RRF_K+r), 'd': d}
    for r, i in enumerate(bm25_top, 1):
        k = chunks[i].page_content[:80]
        if k in rrf: rrf[k]['s'] += 1/(RRF_K+r)
        else: rrf[k] = {'s': 1/(RRF_K+r), 'd': chunks[i]}
    candidates = sorted(rrf.values(), key=lambda x: x['s'], reverse=True)
    return [item['d'] for item in candidates[:FINAL_TOP_K]]

## Grounded Prompt

In [ ]:
GROUNDED_PROMPT = PromptTemplate(
    input_variables=['sources', 'question'],
    template="""You are a precise assistant. Answer ONLY from the provided sources.

Rules:
1. Every claim MUST cite its source as [S1], [S2], etc.
2. Do NOT add information beyond sources.
3. If sources conflict, state both with citations.
4. If insufficient evidence, respond: 'Insufficient evidence in available sources.'
5. Do NOT use prior knowledge.

Sources:
{sources}

Question: {question}

Answer (with citations):"""
)
grounded_chain = GROUNDED_PROMPT | llm

def format_sources(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get('source', 'unknown')
        parts.append(f'[S{i}] (from {src}):\n{d.page_content}')
    return '\n\n'.join(parts)

## Run Evaluation

In [ ]:
all_results = []

for q in eval_questions:
    question = q['question']
    relevant_docs = q.get('relevant_documents', [])
    should_abstain = q.get('should_abstain', False)

    with Timer() as t_ret:
        docs = advanced_retrieve(question)
    retrieved_ids = [d.metadata.get('source', f'c{i}') for i, d in enumerate(docs)]

    sources_text = format_sources(docs)
    with Timer() as t_gen:
        response = grounded_chain.invoke({'sources': sources_text, 'question': question})
    answer = response.content

    predicted_abstain = 'insufficient evidence' in answer.lower()
    has_citation = '[S' in answer

    metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=FINAL_TOP_K)
    metrics['has_citation'] = has_citation
    metrics['predicted_abstain'] = predicted_abstain
    metrics['correct_abstain'] = (predicted_abstain == should_abstain)

    all_results.append(build_result_record(
        experiment_id=EXPERIMENT_ID, notebook=NOTEBOOK,
        config_hash=CONFIG_HASH, seed=SEED,
        question_id=q['question_id'], question=question,
        answer=answer, predicted_abstain=predicted_abstain,
        metrics=metrics,
        latency={'retrieval_seconds': t_ret.elapsed, 'generation_seconds': t_gen.elapsed,
                 'total_seconds': t_ret.elapsed + t_gen.elapsed},
        usage={'context_chars': len(sources_text), 'llm_calls': 1},
    ))
    status = 'ABSTAIN' if predicted_abstain else 'ANSWER'
    print(f'  [{q["question_id"]}] {status} cite={has_citation}')

print(f'\nDone: {len(all_results)} records')

## Summary

In [ ]:
print('='*60)
print(f'A7 ADVANCED RAG ({len(all_results)} questions)')
print('='*60)
cite_rate = np.mean([r['metrics']['has_citation'] for r in all_results])
abstain_acc = np.mean([r['metrics']['correct_abstain'] for r in all_results])
avg_lat = np.mean([r['latency']['total_seconds'] for r in all_results])
print(f'  Citation rate      : {cite_rate:.3f}')
print(f'  Abstention accuracy: {abstain_acc:.3f}')
print(f'  Avg latency        : {avg_lat:.3f}s')
print('='*60)

In [ ]:
output_dir = PROJECT_ROOT / config['paths']['results']
save_jsonl(all_results, output_dir / 'A7_advanced_rag.jsonl')
save_config_snapshot(config, output_dir)
print('Saved.')

## A0 vs A7

| Metric | A0 Naive | A7 Advanced | Delta |
|--------|----------|-------------|-------|
| Recall@5 | | | |
| MRR | | | |
| Citation rate | N/A | | |
| Abstention accuracy | N/A | | |
| Avg latency | | | |